# 933. Number of Recent Calls

**Difficulty:** Easy &nbsp;|&nbsp; **Topics:** design, queue, sliding-window
&nbsp;|&nbsp; [LeetCode](https://leetcode.com/problems/number-of-recent-calls/)

You have a `RecentCounter` class which counts the number of recent requests
within a certain time frame.

Implement the `RecentCounter` class:

- `RecentCounter()` initialises the counter with zero recent requests.
- `ping(t)` adds a new request at time `t` (in milliseconds) and returns the
  number of requests that have happened in the **inclusive** range
  `[t - 3000, t]`.

It is **guaranteed** that every call to `ping` uses a strictly larger value of `t`
than the previous call.

---

### Example

```
Input:  ["RecentCounter", "ping", "ping", "ping", "ping"]
        [[],              [1],    [100],  [3001], [3002]]
Output: [null,            1,      2,      3,      3]

RecentCounter r = new RecentCounter();
r.ping(1);      // requests = [1],                 range is [-2999, 1],  returns 1
r.ping(100);    // requests = [1, 100],            range is [-2900, 100], returns 2
r.ping(3001);   // requests = [1, 100, 3001],      range is [1, 3001],    returns 3
r.ping(3002);   // requests = [1, 100, 3001, 3002], range is [2, 3002],   returns 3
```

---

### Constraints

- `1 <= t <= 10^9`
- Each test case calls `ping` with strictly increasing values of `t`
- At most `10^4` calls will be made to `ping`

This is what sits in front of every API you have ever used: *how many requests has
this client made in the last N seconds?* The whole problem is one sentence in the
constraints, and if you do not notice it you will write an `O(n)`-per-call answer
that still passes.

## Before you write anything

**1.** **Read the guarantee again: `t` is strictly increasing.** That single line
is the problem. Write down, in one sentence, what it lets you *throw away* - and
why you can be certain you will never be asked for it again. Everything below
follows from your answer.

**2.** The version that does not use the guarantee: keep every `t` in a list, and
on each `ping` count how many are `>= t - 3000`. What does one call cost? What do
`10^4` calls cost in total? It is accepted by LeetCode. Say why it is still the
wrong answer to *this* problem.

**3.** The range is `[t - 3000, t]`, **inclusive at both ends**. So a request at
`t = 1` is still counted by a ping at `t = 3001`, and is **not** counted by a ping
at `t = 3002`. Write the discard condition - `<` or `<=`, against `t - 3000` -
and then say what the other one does to the example above.

**4.** With the guarantee, the structure is a **queue**: new times join at one end,
expired times leave at the other. One `ping` can expire a thousand old entries, so
one call is *not* `O(1)`. Yet the total work over `10^4` calls is `O(n)`. Explain
the difference between "`O(1)` per call" and "`O(1)` **amortised** per call" using
the fact that each request is added exactly once and removed at most once.

**5.** In Python, `list.pop(0)` shifts every remaining element down one - `O(n)`.
So a queue built on `pop(0)` turns your amortised `O(1)` back into `O(n)`. Name
the two ways out (one is a `collections` type, one is a plain list plus one extra
integer), and say what the second one leaks.

## Two routes

**A - `collections.deque`** *(write this first)*

```
self.q = deque()
```

`ping(t)`: append `t`, then `popleft()` while the front is older than `t - 3000`,
then return `len(self.q)`. Every request enters once and leaves once, so the total
work across all calls is `O(n)` - amortised `O(1)` each - and the memory is exactly
the number of requests currently inside the window.

`deque` is a doubly linked list of blocks, which is why `popleft` is `O(1)` and
`list.pop(0)` is not. You built the same thing by hand in #707 route B.

**B - a plain list and a head index**

Never remove anything. Keep `self.times` and an integer `self.head`, and advance
`self.head` past the expired entries instead of deleting them. The count is
`len(self.times) - self.head`. Also amortised `O(1)`, with no `deque` import and a
smaller constant.

The price: the list **never shrinks**. After `10^4` pings you are holding `10^4`
timestamps to answer a question about maybe three of them. At LeetCode's scale that
is nothing; in a server that runs for a month it is a memory leak with a slow fuse.
Say out loud which one you would ship, and why the answer depends on how long the
process lives.

> **The guarantee is the algorithm.** Strictly increasing `t` is what turns "search
> a window" into "a queue you only ever push and pop". Read the constraints before
> you design - they are not fine print, they are half the problem statement.

In [ ]:
class RecentCounter:

    def __init__(self):
        pass

    def ping(self, t: int) -> int:
        pass

### The test harness

`ping` returns a number, so wrong answers are visible. The interesting failures are
the boundaries: exactly `3000` apart, exactly `3001` apart, and the very first call
where `t - 3000` is negative.

So `check` replays a list of timestamps against your class and against a brute-force
model - keep every time, count the ones inside the window - and compares call by
call, reporting the first `ping` that disagrees along with the window it was asked
about. The model is deliberately the slow `O(n)` version from question 2: it is
obviously correct, which is exactly what you want from a test oracle and exactly
what you do not want in your answer.

`stress` generates strictly increasing timestamps with a mix of tiny and huge gaps,
so windows both fill up and empty out completely. Run this cell; don't edit it.

In [ ]:
import random


def check(times):
    '''Replay ping(t) for each t against RecentCounter and a brute-force model.'''
    log = []
    try:
        rc = RecentCounter()
    except Exception as e:
        return False, [f"   !! RecentCounter() raised {type(e).__name__}: {e}"]

    seen = []                                   # the obviously-correct model
    for t in times:
        seen.append(t)
        want = sum(1 for x in seen if t - 3000 <= x <= t)
        try:
            got = rc.ping(t)
        except Exception as e:
            log.append(f"   !! ping({t}) raised {type(e).__name__}: {e}")
            return False, log

        log.append(f"ping({t}) -> {got!r}   window [{t - 3000}, {t}]")
        if got != want:
            inside = [x for x in seen if t - 3000 <= x <= t]
            log.append(f"   !! ping({t}) must return {want!r}, got {got!r}")
            log.append(f"      the window [{t - 3000}, {t}] holds {inside[:10]}"
                       f"{' ...' if len(inside) > 10 else ''}")
            return False, log

    return True, log


def stress(n, seed=0, max_gap=2000):
    '''Strictly increasing timestamps with mixed gaps.'''
    random.seed(seed)
    t, times = 1, []
    for _ in range(n):
        times.append(t)
        t += random.randint(1, max_gap)
    return check(times)


def report(name, ok, log, tail=5):
    print(f"{'OK  ' if ok else 'FAIL'} {name}")
    if not ok:
        for line in log[-tail:]:
            print(f"       {line}")

In [ ]:
# tests
CASES = [
    ("the LeetCode example",                  [1, 100, 3001, 3002]),
    ("a single ping",                         [1]),
    ("t = 1, so the window starts negative",  [1, 2, 3]),
    ("question 3: exactly 3000 apart is IN",  [1, 3001]),
    ("question 3: exactly 3001 apart is OUT", [1, 3002]),
    ("every ping 1 ms apart",                 list(range(1, 21))),
    ("every ping far apart - window is always 1",
                                              [1, 5000, 10000, 20000, 100000]),
    ("fill the window, then jump past it",    [1, 2, 3, 4, 5, 100000]),
    ("a dense burst inside one window",       list(range(1000, 1051))),
    ("the ceiling of t",                      [1, 10**9 - 3000, 10**9 - 1, 10**9]),
]

for name, times in CASES:
    report(name, *check(times))

for n, seed, gap in [(50, 1, 100), (200, 2, 2000), (1000, 3, 10), (10000, 4, 700)]:
    report(f"stress: {n} pings (seed {seed}, gaps up to {gap} ms)", *stress(n, seed, gap))

print("\ntrace of the LeetCode example:")
for line in check([1, 100, 3001, 3002])[1]:
    print("  " + line)

## After it passes

- **Prove the amortised bound to yourself.** Add a counter that increments on every
  `popleft`, run the 10 000-ping stress, and print it. It must be at most 10 000 -
  no matter how many any single `ping` removed. That number *is* the proof.
- **Build route B and race it.** Same tests, then `timeit` both at 10 000 pings.
  Then print `len(self.times)` at the end of each and put the two numbers side by
  side. That pair - time equal, memory not - is the whole trade.
- **Break the guarantee.** Feed the timestamps out of order and watch what happens.
  Your answer will be wrong, and it *should* be: you built it on a promise the
  caller made. Now say what you would have to change to survive unordered input,
  and notice that it is a different problem with a different data structure.
- **Make it real.** A rate limiter does not want the count, it wants a yes/no:
  `allow(t)` returns `False` if this would be the 101st request in 3000 ms. That is
  three characters of change and it is the actual production version of this class.
  Then: how do you do it for a million *different* clients without a million deques?
- Siblings: #359 Logger Rate Limiter (the same window, keyed per message),
  #362 Design Hit Counter (the same window, but you are asked for the count without
  a new event), #346 Moving Average from Data Stream (a *fixed-size* window rather
  than a fixed-*time* one - compare which one needs a deque and which does not).